# Draw — Treinamento SegFormer-B0 (4 classes: background / paper / hand / ink)

**Notebook Kaggle oficial do projeto.** Treina SegFormer-B0 com quatro classes, valida, exporta ONNX e registra métricas. Não baixa código nem instala dependências.

O kernel recebe como inputs os seis chunks de treino, os chunks de validação e teste, o código `training/`, o cache de pesos `mit-b0` e o dataset FreiHAND. A célula 5 reúne os chunks por symlink em `/kaggle/working`, sem duplicar os arquivos de imagem.

O treino completo usa o número de épocas de `training/config.json` (30 por padrão). `EPOCHS = 2` na célula 11 pode ser usado manualmente para um smoke test.

## 1. Verificar o ambiente (sem instalar nada)

O Kaggle já traz torch, transformers, numpy e opencv. Este notebook
**não executa `pip install`**: se uma dependência obrigatória faltar,
a célula abaixo levanta um erro claro dizendo qual falta, em vez de
tentar baixar algo sem internet. `albumentations`, `onnx` e `onnxruntime` são obrigatórios. Use as versões
exatas de `training/requirements.txt`; a célula abaixo falha em caso de
dependência ausente ou versão diferente.


In [ ]:
# Verificação do ambiente — SEM instalar pacotes, SEM download.
# Se algo obrigatório faltar, o erro abaixo diz o que falta; não há
# tentativa de baixar pacotes (o kernel pode estar sem internet).
import importlib, os, sys
from pathlib import Path

REQUIRED = ["torch", "transformers", "cv2", "numpy", "albumentations", "onnx", "onnxruntime"]
EXPECTED = {
    "torch": "2.11.0", "transformers": "4.57.1", "cv2": "4.11.0",
    "numpy": "1.26.4", "albumentations": "2.0.8",
    "onnx": "1.23.0", "onnxruntime": "1.31.1",
}
missing, mismatched = [], []
for name in REQUIRED:
    try:
        mod = importlib.import_module(name)
        version = str(getattr(mod, "__version__", "sem versão")).split("+")[0]
        print(f"{name}: {version}")
        if version != EXPECTED[name]:
            mismatched.append(f"{name}=={version} (esperado {EXPECTED[name]})")
    except ImportError:
        missing.append(name)
        print(f"{name}: AUSENTE (obrigatório)")
if missing or mismatched:
    details = []
    if missing: details.append("ausentes: " + ", ".join(missing))
    if mismatched: details.append("versões divergentes: " + "; ".join(mismatched))
    raise RuntimeError("Ambiente incompatível com training/requirements.txt: " + " | ".join(details))

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU: treino completo será muito mais lento; teste pequeno (EPOCHS=2) roda em CPU.")

# Cache dos pesos pré-treinados nvidia/mit-b0 (dataset guzzfort/mit-b0-cache).
# Procura robusta nos dois layouts de input do Kaggle; se achar, fixa o
# cache e liga o modo offline ANTES de qualquer from_pretrained (o train.py
# roda como subprocesso do kernel e herda estas variáveis).
import glob as _glob
_hub_candidates = []
for _pat in ("/kaggle/input/*/hub", "/kaggle/input/datasets/*/*/hub", "/kaggle/input/datasets/*/*/*/hub"):
    _hub_candidates += [d for d in _glob.glob(_pat) if os.path.isdir(os.path.join(d, "models--nvidia--mit-b0"))]
if _hub_candidates:
    os.environ["HF_HUB_CACHE"] = _hub_candidates[0]
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    print("cache de pesos nvidia/mit-b0: encontrado em", _hub_candidates[0], "| modo offline ligado")
else:
    print("AVISO: cache de pesos nvidia/mit-b0 NÃO encontrado. O primeiro from_pretrained "
          "tentará baixar do HuggingFace e falhará sem internet. Para 100% offline, "
          "anexe o dataset guzzfort/mit-b0-cache como input do kernel.")


## 2. Localizar código e dataset (sem clone)

O código `training/` vem como **input do kernel**, não do GitHub.
A célula abaixo procura `training/` nos inputs, nesta ordem:
`/kaggle/input/*/training`, `/kaggle/input/*/draw/training`,
`/kaggle/input/datasets/*/*/training`, `/kaggle/working/draw/training`.
Se não achar, ela **levanta erro e para**
— não segue com um caminho falso.


In [ ]:
# Localização do código training/ — SEM clonagem do repo (modo offline).
# Layouts do Kaggle (PROVADO no reteste): inputs ficam tanto em
# /kaggle/input/<slug>/ quanto em /kaggle/input/datasets/<owner>/<slug>/.
# Por isso procuramos nos DOIS layouts; procurar só no antigo falhava.
from pathlib import Path

code_candidates = []
code_candidates += sorted(Path("/kaggle/input").glob("*/training"))
code_candidates += sorted(Path("/kaggle/input").glob("*/draw/training"))
code_candidates += sorted(Path("/kaggle/input/datasets").glob("*/*/training"))
code_candidates += sorted(Path("/kaggle/input/datasets").glob("*/*/draw/training"))
code_candidates += sorted(Path("/kaggle/input/datasets").glob("*/*/*/training"))
code_candidates.append(Path("/kaggle/working/draw/training"))

REPO = None
for cand in dict.fromkeys(code_candidates):
    if (cand / "train.py").exists():
        REPO = cand.parent
        break
if REPO is None:
    raise RuntimeError(
        "Código training/ NÃO encontrado em nenhum input do kernel. "
        "Anexe o dataset de código com training/train.py + training/config.json "
        "como input do kernel (Add Input). Este notebook NÃO clona o repositório. "
        "Caminhos procurados (layout antigo e novo): /kaggle/input/*/training, "
        "/kaggle/input/*/draw/training, /kaggle/input/datasets/*/*/training, "
        "/kaggle/input/datasets/*/*/draw/training, /kaggle/working/draw/training."
    )
print("repo (código):", REPO, "| train.py:", (REPO / "training" / "train.py").exists())


In [ ]:
# Reúne os chunks Kaggle nos splits train/val/test sem copiar imagens.
# Cada dataset sintético contém parte de um split; a lista completa está em kernel-metadata.json.
from pathlib import Path
import os
import shutil

IMAGE_EXTS = {".png", ".jpg", ".jpeg"}
part_roots = sorted(Path("/kaggle/input/datasets/guzzbr").glob("draw-synthetic-v1-part*/dataset/synthetic"))
if not part_roots:
    part_roots = sorted(Path("/kaggle/input").glob("draw-synthetic-v1-part*/dataset/synthetic"))
if len(part_roots) < 8:
    raise RuntimeError(f"Esperados 8 chunks do dataset sintético; encontrados {len(part_roots)}: {part_roots}")

DATA = Path("/kaggle/working/dataset/synthetic")
if DATA.exists():
    shutil.rmtree(DATA)
for split in ("train", "val", "test"):
    for kind in ("images", "masks", "ink_full"):
        (DATA / split / kind).mkdir(parents=True, exist_ok=True)
    for part in part_roots:
        src_split = part / split
        for kind in ("images", "masks", "ink_full"):
            src_dir = src_split / kind
            if not src_dir.is_dir():
                continue
            for src in sorted(src_dir.iterdir()):
                if src.is_file():
                    dst = DATA / split / kind / src.name
                    if not dst.exists():
                        dst.symlink_to(src)
manifest = next((p / "split_manifest.json" for p in part_roots if (p / "split_manifest.json").is_file()), None)
if manifest is None:
    raise RuntimeError("split_manifest.json não encontrado nos chunks sintéticos")
shutil.copy2(manifest, DATA / "split_manifest.json")

counts = {}
for split in ("train", "val", "test"):
    n = len([f for f in (DATA / split / "images").iterdir() if f.suffix.lower() in IMAGE_EXTS])
    counts[split] = n
    print(f"  {split}: {n} imagens")
missing_splits = [s for s, n in counts.items() if n == 0]
if missing_splits:
    raise RuntimeError(f"Dataset sintético incompleto; splits vazios: {missing_splits}")
print("dataset sintético reunido:", DATA, "| chunks:", len(part_roots))

# Os 719 frames reais são input privado separado. Não execute sem eles:
# não queremos registrar treino como real quando só rodou o sintético.
real_manifests = sorted(Path("/kaggle/input").rglob("dataset_manifest.json"))
REAL_DATA = None
for manifest_path in real_manifests:
    candidate = manifest_path.parent
    if ((candidate / "samples.csv").is_file()
            and (candidate / "train" / "images").is_dir()
            and (candidate / "train" / "masks").is_dir()):
        REAL_DATA = candidate
        break
if REAL_DATA is None:
    raise RuntimeError(
        "Pacote privado pseudo_training_candidate não encontrado. Anexe-o como input do Kaggle; "
        "o notebook não vai omitir os frames reais silenciosamente."
    )
print("pacote real (propostas não revisadas):", REAL_DATA)


## 3. Checagem do FreiHAND (de verdade, não só listagem)

Antes do treino, confirme o que a versão do Kaggle realmente contém:
máscara real de mão em `training/mask/`, com dimensão, limiar 127 e
cobertura. **Se `training/mask/` faltar, esta célula levanta erro e o
notebook para aqui** — não gere rótulos de mão sintéticos fingindo que
são reais, e não siga para o treino sem documentar o problema.
O treino em si usa o dataset sintético; esta célula é a verificação
exigida pelo escopo.


In [ ]:
# Checagem real do FreiHAND — PARA com erro claro se training/mask/ faltar.
# Layouts do Kaggle: /kaggle/input/<slug>/... e /kaggle/input/datasets/<owner>/<slug>/...
import cv2
import numpy as np
from pathlib import Path

mask_dirs = sorted(Path("/kaggle/input").glob("*/training/mask"))
mask_dirs += sorted(Path("/kaggle/input").glob("*/*/training/mask"))
mask_dirs += sorted(Path("/kaggle/input/datasets").glob("*/*/training/mask"))
mask_dirs += sorted(Path("/kaggle/input/datasets").glob("*/*/*/training/mask"))
mask_dirs = [d for d in dict.fromkeys(mask_dirs) if d.is_dir()]
if not mask_dirs:
    raise RuntimeError(
        "FreiHAND: training/mask/ NÃO encontrado em nenhum input do kernel. "
        "Anexe o dataset danieldelro/freihand como input do kernel (Add Input). "
        "Sem máscara real de mão, NÃO prossiga: documente o problema e troque a fonte — "
        "nunca invente rótulos de mão. Caminhos procurados (layout antigo e novo): "
        "/kaggle/input/*/training/mask, /kaggle/input/datasets/*/*/training/mask."
    )
mask_dir = mask_dirs[0]
print("FreiHAND training/mask/:", mask_dir)
mask_files = sorted(mask_dir.glob("*.jpg")) + sorted(mask_dir.glob("*.png"))
if not mask_files:
    raise RuntimeError(f"FreiHAND: nenhum arquivo de máscara em {mask_dir}.")
print(f"exemplo de máscara: {mask_files[0].name} (primeiras {min(5, len(mask_files))} serão amostradas)")
coverages = []
for mf in mask_files[:5]:
    mask = cv2.imread(str(mf), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise RuntimeError(f"FreiHAND: não consegui ler a máscara {mf}.")
    if mask.ndim != 2:
        raise RuntimeError(f"FreiHAND: máscara {mf} não é 2D (shape {mask.shape}).")
    hand = mask > 127  # regra de uso: binarizar com limiar 127, nunca usar o JPEG cru
    cov = float(hand.mean())
    coverages.append(cov)
    rgb = mask_dir.parent / "rgb" / mf.name
    print(f"  {mf.name}: shape {mask.shape}, cobertura mão (limiar 127) {cov:.4f}, RGB pareado: {'OK' if rgb.exists() else 'AUSENTE'}")
if all(c == 0.0 for c in coverages):
    raise RuntimeError("FreiHAND: todas as máscaras amostradas têm cobertura 0 — nenhuma mão detectada. Não prossiga sem investigar.")
if all(c >= 0.99 for c in coverages):
    raise RuntimeError("FreiHAND: todas as máscaras amostradas são quase 100% mão — formato inesperado. Não prossiga sem investigar.")
print("FreiHAND OK para checagem: training/mask/ existe, máscara 2D, limiar 127, cobertura de mão plausível.")


## 4. Config e contagem de pixels

Lê `training/config.json` e conta pixels por classe no treino.
Compare com os pesos da config (CE [1, 2, 3, 10]): se a contagem real
divergir muito, recalibre os pesos e registre a mudança em
`training_config.json` — nunca ajustando peso olhando o split de teste.


In [ ]:
import sys, json
sys.path.insert(0, str(REPO))
from training.train import load_config, pair_images_masks, pixel_counts, compute_class_weights

cfg = load_config(REPO / "training" / "config.json")
print("modelo:", cfg["model"]["architecture"], cfg["model"]["backbone"])
print("classes:", cfg["model"]["classes"])
print("loss:", cfg["loss"]["formula"])
pairs = pair_images_masks(DATA / "train")
counts = pixel_counts(pairs)
print("pixels por classe (treino):", counts.tolist())
print("pesos calculados (referência):", [round(w, 2) for w in compute_class_weights(counts)])
print("pesos da config (usados):", cfg["loss"]["cross_entropy_class_weights"])


## 5. Treino

Chama `training/train.py`, que é a única fonte de verdade do treino
(modelo, loss, métricas, ONNX). O notebook é só um invólucro
reproduzível em cima dele. Os frames reais entram como propostas não verificadas; use o resultado apenas como experimento até revisar as máscaras. Em caso de OOM, o script sugere batch 4
com acumulação — não reduza a resolução: traço fino depende dela.

**Para testar, troque só `EPOCHS` abaixo:** `EPOCHS = None` usa o padrão
da config (30 épocas); `EPOCHS = 2` roda um teste curto. Não edite o
comando de treino.


In [ ]:
# Treino — EPOCHS é a única variável a trocar para testar.
EPOCHS = None  # None = usa a config (30 épocas); para teste, troque para 2

import subprocess, sys
cmd = [
    sys.executable,
    str(REPO / "training" / "train.py"),
    "--config", str(REPO / "training" / "config.json"),
    "--data-dir", str(DATA),
    "--real-train-dir", str(REAL_DATA),
    "--out-dir", "/kaggle/working/out",
]
if EPOCHS is not None:
    cmd += ["--epochs", str(EPOCHS)]
print("comando:", " ".join(cmd))
subprocess.run(cmd, check=True)


## 6. Métricas e cenários fixos

`metrics.json` traz IoU, Dice, precision e recall por classe, a
métrica de ink com tolerância (2-3 px) e a validação ONNX.
**Nunca reporte só accuracy global:** com fundo dominante, ela mente.
Os 15 cenários fixos (folha limpa, mão cobrindo canto/centro/traço,
sombra forte, traço fino de lápis, etc.) têm predições salvas para
comparação entre versões — o avaliador de cenários é o próximo passo
do agente de validação, fora deste notebook.


In [ ]:
import json
from pathlib import Path
metrics_path = Path("/kaggle/working/out/metrics.json")
if not metrics_path.exists():
    raise FileNotFoundError(
        "metrics.json não encontrado em /kaggle/working/out. O treino não completou "
        "ou salvou em outro lugar — verifique a célula de treino antes de ler métricas."
    )
m = json.load(open(metrics_path))
print("val_mean_iou:", round(m["mean_iou"], 4))
for cls in ("background", "paper", "hand", "ink"):
    d = m[cls]
    print(f"{cls:11s} IoU {d['iou']:.3f}  Dice {d['dice']:.3f}  P {d['precision']:.3f}  R {d['recall']:.3f}")
print("validação ONNX:", m.get("onnx_validation"))
print("artefatos:", sorted(p.name for p in Path("/kaggle/working/out").iterdir()))


## 7. Interface ONNX (para o Codex integrar em `draw/neural.py`)

- **Input:** `image`, tensor float32 `[1, 3, 512, 512]`, RGB,
  normalização ImageNet (mean 0.485/0.456/0.406, std 0.229/0.224/0.225)
  — idêntica ao contrato atual de `draw/neural.py`.
- **Output:** `logits`, `[1, 4, 512, 512]`, sem softmax embutido.
  Ordem das classes: 0=background, 1=paper, 2=hand, 3=ink.
- **Uso:** `paper_mask = (argmax == 1)`, `hand_mask = (argmax == 2)`,
  `ink_mask = (argmax == 3)`.
- **paper_presence:** DERIVADO da máscara (fração de pixels paper
  acima do limiar que a integração escolher), não uma cabeça auxiliar.
  O modelo antigo tinha 2 canais + `presence_logit`; o novo tem 4
  canais e cabeça única — a integração muda só a leitura da saída,
  não o pré-processamento.
- **Regra de rótulo:** mão sobre traço = hand. O modelo não enxerga
  através da mão; a memória temporal futura do Draw é que preservará
  o traço escondido, usando `ink_full_mask` (fora do treino) como
  referência de avaliação.
